In [1]:
import wandb
import pandas as pd
import os
from tqdm import tqdm

# from table_plotter import print_result_table

In [2]:
api = wandb.Api(timeout=600)


In [3]:
# Specify cache directory
cache_dir = "./wandb_cache"
os.makedirs(cache_dir, exist_ok=True)

In [4]:

skipped_runs = []  # List to store IDs of skipped runs

evaluation_keys = ['Evaluation/acc_imp_perc', 'Evaluation/exist_imp_perc', 'Evaluation/reach_imp_perc', 'Evaluation/path_length',
                   'Evaluation/fn_imp_perc', 'Evaluation/fp_imp_perc', 'Evaluation/tn_imp_perc', 'Evaluation/tp_imp_perc', 
                   'Evaluation/solvability', 'Evaluation/playability']
evaluation2_keys = ['Evaluation/playability', 'Evaluation/naive_playability', 'Evaluation/solvability', 'Evaluation/acc_imp_perc']


In [5]:
def get_dataframe_from_run(run):
    dfs = list()
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [6]:
def get_dataframe_from_run2(run):
    dfs = []
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation2_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation2_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation2_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [7]:
runs = api.runs("inchangbaek4907/scenario-feedback")
scenario_df = get_dataframe_from_run(runs)
scenario_df = scenario_df[scenario_df['Evaluation/llm_iteration'] <= 6]
# set score column with acc_imp_perc
scenario_df['score'] = scenario_df['Evaluation/acc_imp_perc']
scenario_df

100%|██████████| 62/62 [02:55<00:00,  2.84s/it]

Skipping run ID: lszgyyw3 (state: running)
Skipping run ID: txjaq6tp (state: running)
Skipping run ID: dxviiihu (state: running)
Skipping run ID: 1ircq6g2 (state: running)
Skipping run ID: ga5jqv4x (state: running)
Skipping run ID: j8n2t743 (state: running)
Skipping run ID: 9eb32q3j (state: running)
Skipping run ID: 8s1eyvkq (state: running)
Skipping run ID: txtdbo41 (state: running)
Skipping run ID: zcqdr17b (state: running)
Skipping run ID: 5ta46peu (state: running)


,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/exist_imp_perc,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score
0,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,0.966667,26.344828,0.1,1.933333,0.000000,0.966667,0.966667,0.966667,0.322222
1,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,2.000000,0.000000,1.000000,1.000000,1.000000,0.333333
2,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,2.000000,0.000000,1.000000,1.000000,1.000000,0.333333
3,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.066668,0.0,2.000000,0.000000,1.000000,1.000000,1.000000,0.333333
4,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,2.000000,0.000000,1.000000,1.000000,1.000000,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,73bjh505,finished,2,cot,gpt-4o,2,feedback,hr,6,5,...,1.0,0.983333,26.000000,0.1,0.966667,0.000000,1.933333,0.966667,0.966667,0.644444
302,73bjh505,finished,2,cot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,0.966667,0.033333,2.000000,1.000000,1.000000,0.677778
303,73bjh505,finished,2,cot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,1.000000,0.000000,2.000000,1.000000,1.000000,0.666667
304,73bjh505,finished,2,cot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,0.800000,0.200000,2.000000,1.000000,1.000000,0.733333


In [8]:
runs = api.runs("inchangbaek4907/scenario2-feedback")
scenario2_df = get_dataframe_from_run2(runs)
scenario2_df = scenario2_df[scenario2_df['Evaluation/llm_iteration'] <= 6]
# set score column with playability
scenario2_df['score'] = scenario2_df['Evaluation/playability']
scenario2_df

100%|██████████| 84/84 [00:24<00:00,  3.46it/s] 


,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,reward_feature,fewshot,problem,seed,Evaluation/llm_iteration,Evaluation/playability,Evaluation/naive_playability,Evaluation/solvability,Evaluation/acc_imp_perc,score
0,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,1,0.0,0.000000,0.000000,0.0,0.0
1,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,2,0.0,0.000000,0.000000,0.0,0.0
2,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,3,0.0,0.000000,0.000000,0.0,0.0
3,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,4,0.0,0.000000,0.000000,0.0,0.0
4,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,5,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,6,2,0.0,0.000000,0.000000,0.0,0.0
500,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,6,3,0.0,0.000000,0.000000,0.0,0.0
501,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,6,4,0.0,0.000000,0.000000,0.0,0.0
502,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,6,5,0.0,0.000000,0.000000,0.0,0.0


In [9]:
scenario_df = pd.concat([scenario_df, scenario2_df], ignore_index=True)
scenario_df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
0,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,0.966667,26.344828,0.1,1.933333,0.0,0.966667,0.966667,0.966667,0.322222,NaN
1,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
2,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
3,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.066668,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
4,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
806,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
807,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
808,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000


In [10]:
# Print summary of skipped runs
print("\nSummary of Skipped Runs:")
print(f"Total skipped runs: {len(skipped_runs)}")
print("Skipped run IDs:", skipped_runs)


Summary of Skipped Runs:
Total skipped runs: 0
Skipped run IDs: []


In [11]:
df = pd.concat([scenario_df], ignore_index=True)

In [12]:
df.to_csv(f"feedback_result.csv", index=False)